## Overview

In this script, we randomly sample set of items/videos for analysis.

In [ ]:
from pathlib import Path
from collections import defaultdict
import os, glob, re, random
import pandas as pd
import shutil


curfolder = os.getcwd()
projectdata = os.path.join(curfolder, "ToTrack")

vfiles = glob.glob(os.path.join(projectdata, "*", "*.mp4"), recursive=True)
vfiles = [str(f) for f in vfiles if 'flipped' not in os.path.basename(f).lower()]
print(vfiles)

['g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_(AUF)PUMPEN.mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_ABEND.mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_ABHOLEN.mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_ABLEHNEN.mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_ABTROCKNEN .mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_ADLER.mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_AFFE.mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_AKKORDEON.mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_ALT.mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_ANFANG.mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_ANGELN.mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_ANGENEHM.mp4', 'g:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP01_ANSTECKEN.mp4', 'g:\\GtST_

## Sample without concepts

In [ ]:
TOTAL_PER_GROUP = 250
SEED = 42
random.seed(SEED)

vfiles = glob.glob(os.path.join(projectdata, "*", "*.mp4"), recursive=True)
vfiles = [Path(f) for f in vfiles if 'flipped' not in os.path.basename(f).lower()]

#### Parse filename ####
def parse_filename(stem):
    if stem.startswith("GtST_DGS"):
        return "NativeSigners", "SignerA_DGS"
    if stem.startswith("ViCom"):
        return "NativeSigners", "SignerB_ViCom"
    m = re.match(r"^(LP|IP)(\d+)_(\d+)_", stem)
    if m:
        prefix, pcn, phase = m.groups()
        phase_label = "before" if phase == "01" else "after"
        return f"{prefix}_{phase_label}", f"{prefix}{pcn}"
    m = re.match(r"^GP(\d+)_", stem)
    if m:
        return "Gestures", f"GP{m.group(1)}"
    return None, None

#### Cretae list of files #####
structure = defaultdict(lambda: defaultdict(list))

for f in vfiles:
    group, pcn = parse_filename(f.stem)
    if group is None:
        print(f"Skipping {f.name}: pattern not recognized")
        continue
    structure[group][pcn].append(f)

#### Make sure all groups are up to selected max number ####
rows = []

for group, participants in structure.items():
    remaining = TOTAL_PER_GROUP
    order = sorted(participants, key=lambda p: len(participants[p]))
    n_left = len(order)

    for pcn in order:
        share = remaining // n_left
        take = min(share, len(participants[pcn]))
        chosen = random.sample(participants[pcn], take)
        remaining -= take
        n_left -= 1

        for f in chosen:
            rows.append({"group": group, "pcnID": pcn,
                         "file": f.name, "path": str(f)})

sampled_files = pd.DataFrame(rows)

#### Sanity checks #####
print(sampled_files.groupby("group").size())
print()
print(sampled_files.groupby(["group", "pcnID"]).size())

group
Gestures         250
IP_after         250
IP_before        250
LP_after         250
LP_before        250
NativeSigners    250
dtype: int64

group          pcnID        
Gestures       GP01              17
               GP02              17
               GP03              18
               GP04              18
               GP05              18
                               ... 
LP_before      LP40              11
               LP41              11
               LP42              11
NativeSigners  SignerA_DGS      125
               SignerB_ViCom    125
Length: 67, dtype: int64


In [ ]:
# Save the file
sampled_files.to_csv("sampled_files_noconcept.csv")

## Sample across concepts (not used in this study)

In [8]:
from pathlib import Path
import os, glob

projectdata = os.path.join(os.getcwd(), "ToTrack")
vfiles = glob.glob(os.path.join(projectdata, "*", "*.mp4"), recursive=True)
vfiles = [Path(f) for f in vfiles if 'flipped' not in os.path.basename(f).lower()]

english, german = set(), set()
for f in vfiles:
    s = f.stem
    if s.startswith("GtST_DGS"):
        english.add(s.split("_")[2])
    elif s.startswith("ViCom"):
        english.add(s.split("_")[-1])
    elif s.startswith(("LP", "IP")):
        german.add(s.split("_", 2)[2])
    elif s.startswith("GP"):
        german.add(s.split("_", 1)[1])

print("ENGLISH (signers):")
print(sorted(english))
print("\nGERMAN (LP/IP/GP):")
print(sorted(german))

ENGLISH (signers):
['A-LITTLE', 'A-LOT', 'ACCORDEON', 'ADDICTION', 'ADDRESS', 'AMBULANCE', 'APPLE', 'ARCHITECTURE', 'ARM', 'AUNT', 'AUTOMATICALLY', 'BABY', 'BAD', 'BAPTISM', 'BASKETBALL', 'BEARD', 'BEER', 'BILLIARD', 'BIRD', 'BIRTH', 'BOOK', 'BUTTERFLY', 'CALCULATOR', 'CAMEL', 'CAPPUCINO', 'CAR', 'CENTRAL', 'CITY', 'CLOTHES', 'COAT', 'COLD', 'COMMUNICATION', 'CURIOUS', 'CURTAIN', 'DAUGHTER', 'DAY', 'DEVIL', 'DRESS', 'DRINKINGWATER', 'EGG', 'ELECTRICIAN', 'EMPLOYED', 'EXPERIMENT', 'EYE', 'FATHER', 'FENCE', 'FIRE', 'FLOOR', 'FRAUD', 'GLASSES', 'GOAL', 'HAIR', 'HAMMER', 'HANDS', 'HAT', 'HILL', 'HOBBIES', 'HOCKEY', 'HORSE', 'HOT', 'ID', 'ID-CARD', 'IDEA', 'INTERNET', 'LAND', 'LAPTOP', 'LECTURER', 'LEGS', 'LETTER', 'LIGHTNING', 'MAGAZINE', 'MARRIED', 'MASSAGE', 'MEAT', 'MILK', 'MIXER', 'MOBILE-PHONE', 'MODERATOR', 'MONKEY', 'MONTH', 'MOON', 'MOSQUITO', 'MOTORBIKE', 'NO-MATTER', 'NOSE', 'PAN', 'PARK', 'PARTICIPANT', 'PARTY', 'PAST', 'PASTOR', 'PENGUIN', 'PEPPER', 'PHONE-CALL', 'PHOTOGRAPHY',

In [ ]:
TOTAL_PER_GROUP = 250
SEED = 42
random.seed(SEED)
REQUIRE_BOTH_SIGNERS = True   # concept must exist for signer A AND signer B

curfolder = os.getcwd()
MAPPING_PATH = Path(curfolder) / "signer_concept_map.csv"

#### Parse identifiers ######
def clean(c):
    return c.strip().replace("ẞ", "SS")   # trailing spaces + eszett fix

def parse_filename(stem):
    if stem.startswith("GtST_DGS"):
        return "NativeSigners", "SignerA_DGS", clean(stem.split("_")[2])
    if stem.startswith("ViCom"):
        return "NativeSigners", "SignerB_ViCom", clean(stem.split("_")[-1])
    m = re.match(r"^(LP|IP)(\d+)_(\d+)_(.+)$", stem)
    if m:
        prefix, pcn, phase, concept = m.groups()
        phase_label = "before" if phase == "01" else "after"
        return f"{prefix}_{phase_label}", f"{prefix}{pcn}", clean(concept)
    m = re.match(r"^GP(\d+)_(.+)$", stem)
    if m:
        return "Gestures", f"GP{m.group(1)}", clean(m.group(2))
    return None, None, None

#### Collect files ####
projectdata = os.path.join(curfolder, "ToTrack")
vfiles = glob.glob(os.path.join(projectdata, "*", "*.mp4"), recursive=True)
vfiles = [Path(f) for f in vfiles if 'flipped' not in os.path.basename(f).lower()]

records = []
for f in vfiles:
    group, pcn, concept = parse_filename(f.stem)
    if group is None:
        print(f"Skipping {f.name}: pattern not recognized")
        continue
    records.append((group, pcn, concept, f))

#### load English -> German mapping (blank german rows are excluded) ####
map_df = pd.read_csv(MAPPING_PATH).dropna(subset=["german"])
map_df = map_df[map_df["german"].str.strip() != ""]
en2de = dict(zip(map_df["english"].str.strip(),
                 map_df["german"].str.strip()))

signer_english = sorted({c for g, p, c, f in records if g == "NativeSigners"})
unmapped = [c for c in signer_english if c not in en2de]
if unmapped:
    print(f"Note: {len(unmapped)} signer concepts have no mapping and are excluded:")
    print(f"  {unmapped}")

#### Build structure
structure = defaultdict(lambda: defaultdict(list))
for group, pcn, concept, f in records:
    if group == "NativeSigners":
        concept = en2de.get(concept)
        if concept is None:
            continue
    structure[group][concept].append((pcn, f))

### Anchor concept set on signers
signer_sets = defaultdict(set)
for c, pfs in structure["NativeSigners"].items():
    for pcn, f in pfs:
        signer_sets[pcn].add(c)

if REQUIRE_BOTH_SIGNERS:
    anchor = signer_sets["SignerA_DGS"] & signer_sets["SignerB_ViCom"]
else:
    anchor = signer_sets["SignerA_DGS"] | signer_sets["SignerB_ViCom"]
print(f"\nAnchor (signer) concepts: {len(anchor)}")

shared = set(anchor)
for g in structure:
    if g == "NativeSigners":
        continue
    before = len(shared)
    shared &= set(structure[g])
    print(f"  ∩ {g}: {before} -> {len(shared)}")
shared_concepts = sorted(shared)
print(f"Final shared concept set: {len(shared_concepts)}")

#### Choose the concept set
if len(shared_concepts) >= TOTAL_PER_GROUP:
    chosen_concepts = random.sample(shared_concepts, TOTAL_PER_GROUP)
else:
    chosen_concepts = shared_concepts   # cycle: multiple files per concept

#### Sample per group
rows = []
for group in structure:
    avail = {c: random.sample(structure[group][c], len(structure[group][c]))
             for c in chosen_concepts}
    counts = defaultdict(int)
    selected = []

    while len(selected) < TOTAL_PER_GROUP:
        progress = False
        for c in chosen_concepts:
            if len(selected) >= TOTAL_PER_GROUP:
                break
            if not avail[c]:
                continue
            best = min(avail[c], key=lambda pf: counts[pf[0]])
            avail[c].remove(best)
            pcn, f = best
            selected.append({"group": group, "pcnID": pcn, "concept": c,
                             "file": f.name, "path": str(f)})
            counts[pcn] += 1
            progress = True
        if not progress:
            break
    rows.extend(selected)

sampled_matched = pd.DataFrame(rows)

#### Sannity check ####
print("\nFiles per group:")
print(sampled_matched.groupby("group").size())
print("\nFiles per participant:")
print(sampled_matched.groupby(["group", "pcnID"]).size())
print("\nUnique concepts per group:")
print(sampled_matched.groupby("group")["concept"].nunique())

Note: 40 signer concepts have no mapping and are excluded:
  ['ADDICTION', 'ADDRESS', 'ARCHITECTURE', 'AUNT', 'AUTOMATICALLY', 'BAPTISM', 'CAPPUCINO', 'CITY', 'DAUGHTER', 'DAY', 'ELECTRICIAN', 'EMPLOYED', 'EXPERIMENT', 'FATHER', 'FRAUD', 'HOBBIES', 'INTERNET', 'LAND', 'LECTURER', 'MEAT', 'MODERATOR', 'MONTH', 'PARTICIPANT', 'PROFESSOR', 'SHEEP', 'STORE', 'STUDENT', 'TABLE', 'TO-ADVISE', 'TO-ANNOY', 'TO-CRITICIZE', 'TO-EXPLOIT', 'TO-MOVE', 'TO-SUMMARIZE', 'TO-SUPPLY', 'TO-TEACH', 'TO-WASH-FACE', 'UNCLE', 'WEEK', 'WITNESS']

Anchor (signer) concepts: 139
  ∩ Gestures: 139 -> 83
  ∩ IP_before: 83 -> 79
  ∩ IP_after: 79 -> 79
  ∩ LP_before: 79 -> 79
  ∩ LP_after: 79 -> 79
Final shared concept set: 79

Files per group:
group
Gestures         250
IP_after         250
IP_before        250
LP_after         250
LP_before        250
NativeSigners    158
dtype: int64

Files per participant:
group          pcnID        
Gestures       GP01             18
               GP02             18
        

In [ ]:
# save it
sampled_matched.to_csv("sampled_files_withconcept.csv")

In [ ]:
# how many files each concept contributed, overall
print(sampled_matched["concept"].value_counts())

# per group — this is the important one: rows = concepts, columns = groups
concept_table = sampled_matched.pivot_table(
    index="concept", columns="group", values="file",
    aggfunc="count", fill_value=0
)
print(concept_table)

# summary: is the sampling even across concepts?
print(concept_table.describe())

# concepts with unequal counts across groups (where cycling hit availability limits)
uneven = concept_table[concept_table.nunique(axis=1) > 1]
print(f"\n{len(uneven)} concepts have different counts across groups:")
print(uneven)

concept
ABHOLEN       22
AUSWEIS       22
ABLEHNEN      22
BODEN         22
BILLIARD      22
              ..
MOND          17
MUECKE        17
NEUGIERIG     17
PFEFFER       17
ZERREISSEN    17
Name: count, Length: 79, dtype: int64
group        Gestures  IP_after  IP_before  LP_after  LP_before  NativeSigners
concept                                                                       
ABHOLEN             4         4          4         4          4              2
ABLEHNEN            4         4          4         4          4              2
AFFE                4         4          4         4          4              2
AKKORDEON           4         4          4         4          4              2
ARM                 4         4          4         4          4              2
...               ...       ...        ...       ...        ...            ...
WENIG               3         3          3         3          3              2
WOHNEN              3         3          3         3    

## Prepare files for check

In [ ]:
import shutil

samples1 = pd.read_csv(os.path.join(curfolder, "sampled_files_withconcept.csv"))

copyto = os.path.join(curfolder, "checkfiles")

for row in samples1.iterrows():
    path = row[1]["path"]
    print(path)
    # copy file to copyto folder
    shutil.copy(path, copyto)
    #break


## After check: Identify faulty videos

In [6]:
checked = pd.read_csv(os.path.join(curfolder, "gtst_chefiles_annotation_filled.csv"))
#print(checked.head())

# keep only those where MOVEMENT == 1
checked = checked[checked["MOVEMENT"] == 1]
len(checked)

130

In [9]:
checked

,FILE,WATCHED,CUT,SOUND,HISS,MOVEMENT,NOISE,WEIRD
1922,GP01_AFFE.mp4,1,0,0,0,1,0,0
1924,GP01_ARBEITEN.mp4,1,0,0,0,1,0,0
1925,GP01_ARGUMENTIEREN.mp4,1,0,0,0,1,0,0
1929,GP01_DRECK.mp4,1,0,0,0,1,0,0
1936,GP01_INSEL.mp4,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...
4797,LP38_01_AUSRADIEREN.mp4,1,0,0,0,1,0,0
4811,LP38_01_VORHANG .mp4,1,0,0,0,1,0,0
4875,LP41_01_HOCKEY.mp4,1,0,0,0,1,0,0
4967,LP42_02_PINGUIN.mp4,1,0,0,0,1,0,0


In [13]:
sample1 = pd.read_csv(os.path.join(curfolder, "sampled_files_noconcept.csv"))

# print which files are in sample1 and in checked
replace1 = sample1[sample1["file"].isin(checked["FILE"])]
print(len(replace1))
replace1

74


,Unnamed: 0,group,pcnID,file,path
0,0,Gestures,GP01,GP01_UNTERSTUETZEN.mp4,f:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_t...
1,1,Gestures,GP01,GP01_DRECK.mp4,f:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_t...
3,3,Gestures,GP01,GP01_INSEL.mp4,f:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_t...
12,12,Gestures,GP01,GP01_NEID.mp4,f:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_t...
13,13,Gestures,GP01,GP01_ARGUMENTIEREN.mp4,f:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_t...
...,...,...,...,...,...
982,982,LP_before,LP41,LP41_01_HOCKEY.mp4,f:\GtST_FLESH_KinTrajectory\ToTrack\LP\LP41_01...
1063,1063,LP_after,LP17,LP17_02_PINGUIN.mp4,f:\GtST_FLESH_KinTrajectory\ToTrack\LP\LP17_02...
1084,1084,LP_after,LP18,LP18_02_SPINNE.mp4,f:\GtST_FLESH_KinTrajectory\ToTrack\LP\LP18_02...
1088,1088,LP_after,LP29,LP29_02_PINGUIN.mp4,f:\GtST_FLESH_KinTrajectory\ToTrack\LP\LP29_02...


In [ ]:
replacewith = ["GP01_ADLER.mp4", "GP01_BLITZ.mp4", "GP01_GEIGE.mp4", "GP01_KLOPFEN.mp4", "GP01_MENSCH.mp4", "GP01_RUFEN.mp4", "GP02_IGNORIEREN.mp4", "GP03_ALT.mp4", "GP03_PAPIER.mp4", "GP03_KOKOSNUSS.mp4", "GP04_ELEFANT.mp4", "GP04_UNTERGEHEN.mp4", "GP04_ZIEHEN.mp4", "GP04_PUDERN.mp4", "GP04_OHRFEIGEN.mp4", "GP04_LEIHEN.mp4", "GP05_COWBOY.mp4", "GP05_EISCREME.mp4", "GP05_FALTEN.mp4", "GP05_GOTT.mp4", "GP05_IDEE.mp4", "GP05_PILZ.mp4", "GP06_LOEWE.mp4", "GP06_UNIVERSITAET.mp4", "GP06_SOLDAT.mp4", "GP10_MESSEN.mp4", "GP10_ENTE.mp4", "GP10_RASSEL.mp4", "GP10_DECKE.mp4", "GP10_SCHOEN.mp4", "GP10_BISS.mp4", "GP10_MUELL.mp4", "GP11_ABEND.mp4", "GP11_HASE.mp4", "GP11_RUDERN.mp4", "GP14_TREPPE.mp4", "GP16_PILOT.mp4", "GP16_HUSTEN.mp4", "GP16_TELEFON.mp4", "GP16_NUSS.mp4", "GP16_WINDRAD.mp4", "GP16_SKI FAHREN.mp4", "GP17_DOLMETSCHEN.mp4", "GP17_URLAUB.mp4", "GP18_INSEL.mp4", "IP06_01_KRIEG.mp4", "IP18_01_VIDEOSPIEL.mp4", "IP06_02_SCHLANGE.mp4", "IP06_02_SANDWICH.mp4", "IP06_02_AUGE.mp4", "IP06_02_AUTO.mp4", "IP06_02_KOMMUNIKATION.mp4", "IP06_02_LIPPENSTIFT AUFTRAGEN.mp4", "IP06_02_BASKETBALL.mp4", "IP11_02_HAMMER.mp4", "IP16_02_KLAVIER.mp4", "IP16_02_TENNIS.mp4", "IP16_02_SPINNE.mp4", "IP16_02_MANTEL.mp4", "IP16_02_VERHEIRATET.mp4", "IP16_02_HANDY.mp4", "LP04_01_LOEFFEL.mp4", "LP19_01_KOTZEN.mp4", "LP19_01_KRIEG.mp4", "LP19_01_MOTORRAD.mp4", "LP19_01_HEISS.mp4", "LP22_01_ARGUMENTIEREN.mp4", "LP25_01_AUGE.mp4", "LP41_01_HAARE.mp4", "LP17_02_PFLUECKEN.mp4", "LP18_02_SCHAMPOO.mp4", "LP29_02_PFANNE.mp4", "LP36_02_KLATSCHEN.mp4"]

# from sample1, delete rows from replace1, and create new rows with replacewith
new_sample1 = sample1[~sample1["file"].isin(replace1["file"])]

# add new row for each item from replace with (value into column file), the rest left NA
new_rows = pd.DataFrame({"file": replacewith})
new_sample1 = pd.concat([new_sample1, new_rows], ignore_index=True)

In [ ]:
for index, row in new_sample1.iterrows():
    if pd.notna(row["path"]):
        print("already done")
        print(row["path"])
        continue
    else:
        # Find the filename in vfiles and get the full path into the column path
        path = [x for x in vfiles if row["file"] in x]
        if path:
            new_sample1.at[index, "path"] = path[0]
            print(path[0])
        else:
            print(f"File not found: {row['file']}")

## Copy for Sapiens

In [ ]:
copyto = f"E:\sapiens2\gtst" # This is where our Sapiens installation lives

for row in new_sample1.iterrows():
    path = row[1]["path"]
    print(path)
    shutil.copy(path, copyto)
